# Convolutional Neural Networks (CNN) – Beginner-Friendly Python Implementation

## Image Classification using CIFAR-10

**Goal:** Build a CNN that learns to classify images into 10 categories: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, and truck.

### What you will learn
- Load and explore an image dataset
- Understand image dimensions and pixel values
- Prepare labels for a neural network
- Build a CNN using convolution, pooling, flattening and dense layers
- Compile, train and evaluate the model
- Make predictions and understand the results

> **How to use this notebook:** Read the explanation above each code cell, run the code, and inspect the output before moving forward. You do not need to memorize the code.

## 1. What are we building?

Imagine showing a computer thousands of pictures of cats, dogs, cars and other objects. We also tell it the correct name for each picture. The computer learns visual patterns from these examples and later tries to identify objects in pictures it has never seen.

A **Convolutional Neural Network (CNN)** is a type of deep-learning model designed especially for image-related tasks.

A CNN commonly uses these steps:

1. **Convolution:** Looks for useful patterns such as edges, curves and textures.
2. **Activation (ReLU):** Adds the ability to learn complex patterns.
3. **Pooling:** Reduces the size of feature maps while retaining useful information.
4. **Flatten:** Converts the extracted features into a one-dimensional list.
5. **Dense layers:** Use those features to make a final classification.
6. **Softmax:** Produces a probability for each class.

We will see each of these in the code.

## 2. Import the required libraries

A library is a collection of ready-made Python tools. Instead of writing everything ourselves, we import tools that perform common tasks.

We will use TensorFlow/Keras to build the CNN, NumPy to work with arrays, and Matplotlib to display images.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.utils import to_categorical

print("TensorFlow version:", tf.__version__)

### Understand the import statements

- `import numpy as np`: imports NumPy and gives it the short name `np`. NumPy helps us work with numerical arrays.
- `import matplotlib.pyplot as plt`: imports plotting functions and gives them the short name `plt`.
- `import tensorflow as tf`: imports TensorFlow, a deep-learning library.
- `from tensorflow.keras... import ...`: imports specific Keras tools that we need to create and train the model.
- `print(...)`: displays information in the output area.

**If TensorFlow is not installed:** run `!pip install tensorflow` in a separate notebook cell, then restart the kernel and run the imports again. In Google Colab, TensorFlow is usually already available.

## 3. Load the CIFAR-10 dataset

CIFAR-10 is a commonly used image dataset. It contains 60,000 colour images divided into 10 categories.

- 50,000 images are provided for training.
- 10,000 images are provided for testing.
- Each image is 32 pixels wide and 32 pixels high.
- Each image has 3 colour channels: Red, Green and Blue (RGB).

The dataset function returns four arrays:
- `x_train`: training images
- `y_train`: correct labels for training images
- `x_test`: testing images
- `y_test`: correct labels for testing images

In [ ]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)
print("Testing images:", x_test.shape)
print("Testing labels:", y_test.shape)

### Understand the output and the word `shape`

An array's `shape` tells us its dimensions.

For example, a training image array usually has this shape:

`(50000, 32, 32, 3)`

- `50000`: number of images
- `32`: image height in pixels
- `32`: image width in pixels
- `3`: number of colour channels (RGB)

The labels usually have shape `(50000, 1)`: one class number for each training image.

**Important:** `x` represents input images, while `y` represents the correct answers (labels).

## 4. Understand the 10 image categories

The dataset uses numbers from 0 to 9 to represent the image categories. We create a Python list so that we can display the category names instead of only numbers.

A **list** is an ordered collection of values. Python uses square brackets `[]` to create a list.

In [ ]:
class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("Number of classes:", len(class_names))
print("Class at index 0:", class_names[0])
print("Class at index 5:", class_names[5])

`len(class_names)` counts the number of items in the list. `class_names[0]` accesses the first item because Python indexing starts at zero.

The class number `5`, for example, represents a dog.

## 5. Display some training images

Before building a model, it is useful to look at the data. This is called **data exploration**.

- `plt.figure(figsize=(12, 6))` creates a plotting area.
- `plt.subplot(rows, columns, position)` selects a location in a grid.
- `plt.imshow(image)` displays an image.
- `plt.title(...)` adds a title.
- `plt.axis("off")` hides the axis markings.

In [ ]:
plt.figure(figsize=(12, 6))

for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_train[i])
    label_number = int(y_train[i][0])
    plt.title(class_names[label_number])
    plt.axis("off")

plt.tight_layout()
plt.show()

### Understand the `for` loop

`for i in range(12):` repeats the indented instructions 12 times. The variable `i` takes the values 0, 1, 2, and so on up to 11.

`y_train[i][0]` gets the class number for image `i`. We use `int(...)` to convert it into a standard Python integer so it can be used as a list index.

Try changing `12` to `20` and adjust the subplot grid to `4, 5` to display 20 images.

## 6. Normalize the image pixel values

A digital image is made up of pixels. In this dataset, each RGB pixel channel has a value between **0 and 255**.

- `0` means no intensity for that channel.
- `255` means maximum intensity for that channel.

Neural networks often train more effectively when input values are on a smaller, consistent scale. We divide every pixel value by 255, changing the range from 0–255 to 0–1.

`astype("float32")` changes the array's data type to 32-bit decimal numbers. This is useful because division produces decimal values and neural-network calculations commonly use floating-point numbers.

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print("Minimum pixel value:", x_train.min())
print("Maximum pixel value:", x_train.max())

### Why do we use `255.0` instead of `255`?

Both perform the same mathematical scaling here. Writing `255.0` makes it clear that we are using a decimal number. After normalization, pixel values are between 0 and 1.

**Remember:** We normalize the input images (`x_train` and `x_test`). We do not divide the class labels by 255.

## 7. Convert class labels into one-hot encoding

The original labels are integers from 0 to 9. For example, class `2` represents a bird.

In this notebook, we use `categorical_crossentropy` as the loss function. It expects labels in one-hot encoded form.

One-hot encoding turns a class number into a row of ten values. For example:

- Class 2 becomes `[0, 0, 1, 0, 0, 0, 0, 0, 0, 0]`
- Class 5 becomes `[0, 0, 0, 0, 0, 1, 0, 0, 0, 0]`

Only the position representing the correct class contains `1`; the other positions contain `0`.

In [ ]:
number_of_classes = 10

y_train = to_categorical(y_train, number_of_classes)
y_test = to_categorical(y_test, number_of_classes)

print("One-hot encoded training labels shape:", y_train.shape)
print("Example encoded label:", y_train[0])

### Understand `to_categorical()`

`to_categorical(labels, number_of_classes)` converts integer class labels into one-hot encoded rows.

- `labels`: the original class numbers
- `number_of_classes`: the total number of possible categories (10 here)

The label array changes from a shape like `(50000, 1)` to `(50000, 10)`.

## 8. Create a validation dataset

We use three different sets of data for different purposes:

| Dataset | Purpose |
|---|---|
| Training set | Used by the model to learn |
| Validation set | Used during training to check how well the model is performing on unseen examples |
| Test set | Used at the end for a final evaluation |

We will take the first 5,000 examples from the original training data for validation. The remaining 45,000 examples will be used for training.

The test set remains separate.

In [ ]:
x_valid = x_train[:5000]
y_valid = y_train[:5000]

x_train = x_train[5000:]
y_train = y_train[5000:]

print("Training images:", x_train.shape[0])
print("Validation images:", x_valid.shape[0])
print("Testing images:", x_test.shape[0])

### Understand slicing

A slice selects part of a sequence or array.

- `x_train[:5000]` selects items from the beginning up to (but not including) index 5000.
- `x_train[5000:]` selects items from index 5000 to the end.
- The colon `:` means a range of positions.

We apply the same split to images and labels so that each image stays paired with its correct label.

## 9. Build the CNN model

Now we create the neural network.

We use `Sequential`, which means the layers are arranged one after another. Data enters the first layer, passes through each layer in order, and reaches the output layer.

### The architecture

1. Convolution layer – 16 filters
2. Max-pooling layer
3. Convolution layer – 32 filters
4. Max-pooling layer
5. Convolution layer – 64 filters
6. Max-pooling layer
7. Dropout
8. Flatten
9. Dense layer – 128 neurons
10. Dropout
11. Output layer – 10 neurons

The numbers of filters and neurons are design choices. They are not the number of image categories.

In [ ]:
model = Sequential()

# First convolution block
model.add(Conv2D(
    filters=16,
    kernel_size=(3, 3),
    padding="same",
    activation="relu",
    input_shape=(32, 32, 3)
))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Second convolution block
model.add(Conv2D(
    filters=32,
    kernel_size=(3, 3),
    padding="same",
    activation="relu"
))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Third convolution block
model.add(Conv2D(
    filters=64,
    kernel_size=(3, 3),
    padding="same",
    activation="relu"
))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Classification part
model.add(Dropout(0.3))
model.add(Flatten())
model.add(Dense(128, activation="relu"))
model.add(Dropout(0.3))
model.add(Dense(10, activation="softmax"))

model.summary()

## 10. Understand every CNN layer and parameter

### A. `Sequential()`
Creates a model in which layers are stacked in order.

### B. `Conv2D()`
A convolution layer uses small filters (also called kernels) to scan an image or feature map and detect patterns.

| Parameter | Meaning |
|---|---|
| `filters=16` | Learns 16 different pattern detectors in this layer |
| `kernel_size=(3, 3)` | Each filter looks at a 3 × 3 area at a time |
| `padding="same"` | Adds padding so the output keeps the same height and width as the input to this layer when stride is 1 |
| `activation="relu"` | Applies the ReLU activation function |
| `input_shape=(32, 32, 3)` | Input image height, width and colour channels; needed on the first layer |

**ReLU** means Rectified Linear Unit. It changes negative values to zero and keeps positive values unchanged.

### C. `MaxPooling2D(pool_size=(2, 2))`
Looks at each 2 × 2 area and keeps its maximum value. This reduces the feature-map width and height, helping the network process less information.

### D. `Dropout(0.3)`
During training, randomly switches off 30% of the layer's outputs for that training step. This helps reduce overfitting. Dropout is not applied in the same way during normal prediction.

### E. `Flatten()`
Converts the multi-dimensional feature maps into a one-dimensional sequence of numbers. It does not learn filters; it reshapes the data.

### F. `Dense(128, activation="relu")`
A fully connected layer with 128 neurons. Each neuron receives input from all outputs of the previous layer.

### G. `Dense(10, activation="softmax")`
The final layer has 10 output neurons because there are 10 classes. Softmax converts the outputs into class probabilities that add up to approximately 1.

### Why do the filter counts increase?

We use 16, then 32, then 64 filters as one possible design. Earlier layers can learn simpler patterns; deeper layers can combine information into more complex patterns. Increasing filters gives later layers more pattern detectors, but also increases computation.

## 11. Understand the model summary

The `model.summary()` output shows:
- **Layer (type):** the name and kind of each layer
- **Output Shape:** the shape of the data after that layer
- **Param #:** the number of trainable and non-trainable parameters in that layer

A parameter is a value the network can learn, such as a weight or bias. During training, the model adjusts its trainable parameters to reduce prediction errors.

Look at the output shapes. Pooling reduces the height and width. `Flatten` turns the feature maps into a one-dimensional vector, and the final Dense layer produces 10 outputs.

## 12. Compile the model

Before training, we must configure the model. This is called **compiling**.

The three important choices are:

- **Loss function:** measures how far the model's predictions are from the correct answers.
- **Optimizer:** updates the model's learnable parameters to reduce the loss.
- **Metrics:** values we want to monitor, such as accuracy.

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

### Understand each argument

- `optimizer="adam"`: Adam is an optimization algorithm that adjusts the model's weights during training.
- `loss="categorical_crossentropy"`: a loss function commonly used for multi-class classification when labels are one-hot encoded.
- `metrics=["accuracy"]`: asks Keras to report the percentage of predictions that are correct.

**Loss and accuracy are not the same:** loss measures prediction error using the selected loss formula; accuracy measures the fraction of correct class predictions.

## 13. Save the best model during training

A **callback** is a function that Keras can run at specific points during training.

`ModelCheckpoint` can save the model weights when a monitored value improves. Here, we save the weights with the best validation accuracy.

- `filepath`: where to save the weights
- `monitor`: which value to watch
- `mode="max"`: higher accuracy is better
- `save_best_only=True`: save only when the monitored value improves

In [ ]:
checkpoint = ModelCheckpoint(
    filepath="best_cnn.weights.h5",
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    save_weights_only=True,
    verbose=1
)

## 14. Train the CNN model

Training is the process in which the model sees training images, makes predictions, calculates loss, and updates its weights.

Important arguments in `fit()`:

| Argument | Meaning |
|---|---|
| `x_train` | Training images |
| `y_train` | Correct training labels |
| `batch_size=32` | Number of images processed in one batch |
| `epochs=5` | Number of complete passes through the training set |
| `validation_data` | Data used to check performance after each epoch |
| `callbacks` | Extra actions to perform during training |
| `shuffle=True` | Shuffles training examples between epochs |
| `verbose=1` | Displays training progress |

We use 5 epochs here to keep the demonstration manageable. Training longer may improve results, but it is not guaranteed. Actual accuracy depends on the model, training time and hardware.

In [ ]:
history = model.fit(
    x_train,
    y_train,
    batch_size=32,
    epochs=5,
    validation_data=(x_valid, y_valid),
    callbacks=[checkpoint],
    shuffle=True,
    verbose=1
)

### Understand the training output

You will see values such as:

- `loss`: training loss for the epoch
- `accuracy`: training accuracy for the epoch
- `val_loss`: validation loss
- `val_accuracy`: validation accuracy

An **epoch** is one complete pass through all training examples. A **batch** is a smaller group of examples processed together.

Do not expect 100% accuracy. The goal is to learn patterns that generalize to images the model has not seen.

## 15. Plot training and validation performance

The `history` variable stores the measurements collected during training. We can use it to draw graphs.

- `history.history["accuracy"]`: training accuracy for each epoch
- `history.history["val_accuracy"]`: validation accuracy for each epoch
- `history.history["loss"]`: training loss for each epoch
- `history.history["val_loss"]`: validation loss for each epoch

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()

plt.tight_layout()
plt.show()

### How to read these graphs

- If training and validation accuracy both improve, the model may be learning useful patterns.
- If training accuracy keeps increasing but validation accuracy stops improving or decreases, the model may be overfitting.
- If both accuracies are low, the model may not have learned enough yet.
- Lower loss is generally better, but compare training and validation loss together.

These are clues, not absolute rules. A model's performance should also be checked on the separate test set.

## 16. Load the best saved weights

During training, the checkpoint saved the weights whenever validation accuracy improved.

We now load those best saved weights before evaluating the model. This ensures that we evaluate the best checkpoint rather than automatically using only the final epoch's weights.

In [ ]:
model.load_weights("best_cnn.weights.h5")
print("Best saved weights loaded successfully.")

## 17. Evaluate the model on test data

The test dataset was kept separate from training and validation. We use it now to estimate how well the model performs on unseen images.

`model.evaluate()` returns the loss and the metric values in the same order used during compilation. Since we compiled with loss first and accuracy second, `test_results[0]` is loss and `test_results[1]` is accuracy.

In [ ]:
test_results = model.evaluate(x_test, y_test, verbose=1)

test_loss = test_results[0]
test_accuracy = test_results[1]

print("Test loss:", test_loss)
print("Test accuracy: {:.2f}%".format(test_accuracy * 100))

### Understand the result

If the accuracy is `0.70`, that means approximately 70% of the test images were classified correctly.

We multiply by 100 to display accuracy as a percentage. Your exact result may differ depending on training conditions and software versions.

## 18. Make predictions on test images

Now the model can make predictions.

`model.predict(x_test)` returns a set of 10 scores for each image. Because our final layer uses softmax, these scores represent class probabilities.

For one image:
- `predictions[index]` gives the 10 class probabilities.
- `np.argmax(...)` returns the position of the largest value.
- That position identifies the predicted class.

In [ ]:
predictions = model.predict(x_test)

print("Prediction array shape:", predictions.shape)
print("Probabilities for the first test image:")
print(predictions[0])
print("Sum of probabilities:", predictions[0].sum())

## 19. Display predictions alongside the actual answers

We will display a selection of test images. Each title shows:

- **Predicted:** the category selected by the model
- **Actual:** the correct category from the dataset

A blue title means the prediction is correct. A red title means the prediction is incorrect.

In [ ]:
plt.figure(figsize=(14, 10))

random_indices = np.random.choice(
    len(x_test),
    size=20,
    replace=False
)

for position, image_index in enumerate(random_indices):
    plt.subplot(4, 5, position + 1)

    plt.imshow(x_test[image_index])

    predicted_class = np.argmax(predictions[image_index])
    actual_class = np.argmax(y_test[image_index])

    predicted_name = class_names[predicted_class]
    actual_name = class_names[actual_class]

    title_color = "blue" if predicted_class == actual_class else "red"

    plt.title(
        "Predicted: " + predicted_name + "\nActual: " + actual_name,
        color=title_color,
        fontsize=9
    )
    plt.axis("off")

plt.tight_layout()
plt.show()

### Understand the important NumPy functions

- `np.random.choice(...)`: randomly selects image positions.
- `len(x_test)`: counts the number of test images.
- `size=20`: selects 20 positions.
- `replace=False`: does not select the same position more than once.
- `np.argmax(values)`: returns the index of the largest value.
- `enumerate(...)`: gives both the loop count (`position`) and the selected value (`image_index`).

The model may make mistakes. That is normal. Look at the incorrect examples and consider which categories look visually similar.

## 20. Predict one image and inspect its probabilities

Let's inspect one test image in detail. We will display the image, the predicted category, and the probability assigned to every category.

In [ ]:
image_index = 0

single_image = x_test[image_index]
actual_class = np.argmax(y_test[image_index])

# Add a batch dimension because the model expects a batch of images.
image_batch = np.expand_dims(single_image, axis=0)

# Predict the class probabilities.
single_prediction = model.predict(image_batch, verbose=0)[0]

predicted_class = np.argmax(single_prediction)

plt.imshow(single_image)
plt.axis("off")
plt.title(
    "Predicted: " + class_names[predicted_class] +
    " | Actual: " + class_names[actual_class]
)
plt.show()

for class_index, probability in enumerate(single_prediction):
    print(
        class_names[class_index],
        ":", "{:.2f}%".format(probability * 100)
    )

### Why do we use `np.expand_dims()`?

The model expects image data in batches. Even if we want to predict just one image, we must provide a batch dimension.

A single image has shape `(32, 32, 3)`. After `np.expand_dims(single_image, axis=0)`, its shape becomes `(1, 32, 32, 3)`.

The `1` means that the batch contains one image.

## 21. Important terms – quick revision

| Term | Beginner-friendly meaning |
|---|---|
| CNN | A neural network designed to learn patterns in images |
| Pixel | A tiny unit of a digital image |
| RGB | Red, Green and Blue colour channels |
| Dataset | A collection of examples used for learning or evaluation |
| Feature | A useful pattern or characteristic in data |
| Filter / Kernel | A small matrix that scans an image or feature map |
| Convolution | The operation that applies a filter to an image or feature map |
| Feature map | The output produced by a filter |
| Padding | Extra border values added around the input |
| Stride | How far a filter moves at each step |
| ReLU | An activation function that changes negative values to zero |
| Pooling | A method for reducing feature-map size |
| Flatten | Converts multi-dimensional data into a one-dimensional sequence |
| Dense layer | A fully connected neural-network layer |
| Dropout | Randomly switches off some outputs during training to help reduce overfitting |
| Softmax | Converts output scores into class probabilities |
| Epoch | One complete pass through the training dataset |
| Batch size | Number of examples processed together |
| Loss | A numerical measure of prediction error |
| Optimizer | The method used to update model weights |
| Accuracy | The fraction of predictions that are correct |
| Overfitting | When a model learns training examples too closely and performs worse on new examples |
| Validation set | Data used to monitor performance during model development |
| Test set | Separate data used for final evaluation |

## 22. Practice exercises for students

Try these changes one at a time. Run the notebook again and observe what changes.

1. Change the first convolution layer from `filters=16` to `filters=32`. What changes in the model summary?
2. Change `kernel_size=(3, 3)` to `kernel_size=(5, 5)`. What does this change mean?
3. Change `Dropout(0.3)` to `Dropout(0.5)`. What does the dropout rate mean?
4. Change the batch size from `32` to `64`.
5. Change the number of epochs from `5` to `10`.
6. Remove one pooling layer and inspect the output shapes.
7. Display 20 images instead of 12 in the visualization section.

**Important:** Change one thing at a time. Otherwise, it becomes difficult to understand which change affected the result.

## 23. Final understanding

You have completed an end-to-end image-classification workflow:

**Load data → Explore images → Normalize pixels → Encode labels → Split data → Build CNN → Compile → Train → Save best weights → Evaluate → Predict**

The key idea is that we do not manually tell the CNN what a cat's ears, a dog's nose, or a car's wheels look like. The CNN learns useful patterns from labelled examples during training.

**Teaching tip:** Ask students to explain the purpose of each stage in their own words before asking them to memorize the syntax.